# PyTorch fundamentals

So far, we have built everything from scratch: manual gradients, a custom autodiff engine, and hand-coded neural networks. This is great for understanding, but in practice we use **PyTorch**, a framework that handles all of this for us.

In this guided exercise, we solve the **same problem four times** at increasing levels of abstraction. The problem: fit a third-order polynomial 

$$y = a + bx + cx^2 + dx^3$$

to the sine function.

Each level replaces some of our manual work with PyTorch functionality. By the end, you'll see how the training loop we built by hand maps directly onto PyTorch's API.

In [ ]:
import numpy as np
import torch
import math
import matplotlib.pyplot as plt

## Level 1: NumPy (manual everything)

This is what we've been doing: forward pass, manual gradient computation, manual weight update. Just **read and run** this cell — no changes needed.

In [ ]:
# Generate data
x_np = np.linspace(-math.pi, math.pi, 2000)
y_np = np.sin(x_np)

# Random initial weights
a = np.random.randn()
b = np.random.randn()
c = np.random.randn()
d = np.random.randn()

learning_rate = 1e-6
for t in range(2000):
    # Forward pass
    y_pred = a + b * x_np + c * x_np**2 + d * x_np**3

    # Loss
    loss = np.square(y_pred - y_np).sum()

    # Backward pass: manual gradient computation
    grad_a = (2 * (y_pred - y_np)).sum()
    grad_b = (2 * (y_pred - y_np) * x_np).sum()
    grad_c = (2 * (y_pred - y_np) * x_np**2).sum()
    grad_d = (2 * (y_pred - y_np) * x_np**3).sum()

    # Update weights
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

print(f"Result: y = {a:.4f} + {b:.4f}x + {c:.4f}x² + {d:.4f}x³")
plt.plot(x_np, y_np, label="sin(x)")
plt.plot(x_np, a + b * x_np + c * x_np**2 + d * x_np**3, label="Fitted polynomial")
plt.legend()
plt.title("Level 1: NumPy")
plt.show()

## Level 2: PyTorch tensors

PyTorch tensors are like NumPy arrays, but they can run on GPUs and (as we'll see next) track gradients automatically.

**Your task:** Convert the NumPy code above to use PyTorch tensors. Replace `np.linspace` → `torch.linspace`, `np.sin` → `torch.sin`, `np.random.randn` → `torch.randn`, and `np.square` → `torch.square`. The rest of the logic stays identical.

In [ ]:
# TODO: Convert to PyTorch tensors
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

a = ...  # TODO: torch.randn(())
b = ...
c = ...
d = ...

learning_rate = 1e-6
for t in range(2000):
    y_pred = ...  # TODO: same formula, but with tensors
    loss = ...  # TODO: same loss

    # Manual gradients (same formulas)
    grad_a = ...
    grad_b = ...
    grad_c = ...
    grad_d = ...

    # Update
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

print(f"Result: y = {a:.4f} + {b:.4f}x + {c:.4f}x² + {d:.4f}x³")
plt.plot(x, y, label="sin(x)")
plt.plot(x, a + b * x + c * x**2 + d * x**3, label="Fitted polynomial")
plt.legend()
plt.title("Level 2: PyTorch tensors")
plt.show()

## Level 3: Autograd

Now we let PyTorch compute the gradients **automatically** — just like our `Variable` class, but much more powerful.

The key changes:
1. Create tensors with `requires_grad=True` — this tells PyTorch to track operations
2. Call `loss.backward()` — this computes all gradients automatically
3. Access gradients via `a.grad`, `b.grad`, etc.

**Your task:** Delete the manual gradient computation. Add `requires_grad=True` to the weight tensors, call `loss.backward()`, and use `.grad` for the updates.

In [ ]:
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

# TODO: add requires_grad=True
a = torch.randn(())
b = torch.randn(())
c = torch.randn(())
d = torch.randn(())

learning_rate = 1e-6
for t in range(2000):
    y_pred = a + b * x + c * x**2 + d * x**3
    loss = (y_pred - y).pow(2).sum()

    # TODO: call loss.backward() instead of computing gradients manually

    with torch.no_grad():
        # TODO: update using a.grad, b.grad, etc.
        # Then zero the gradients with a.grad = None, etc.
        pass

print(f"Result: y = {a:.4f} + {b:.4f}x + {c:.4f}x² + {d:.4f}x³")

Now try plotting the result the same way we did in Level 1 and 2:

In [ ]:
plt.plot(x, y, label="sin(x)")
plt.plot(x, a + b * x + c * x**2 + d * x**3, label="Fitted polynomial")
plt.legend()
plt.title("Level 3: Autograd")
plt.show()

This will raise an error. The problem is that `a`, `b`, `c`, `d` have `requires_grad=True`, which means PyTorch is tracking every operation involving them to build the computational graph. Matplotlib doesn't know how to handle tensors that are part of a computation graph — it needs plain numbers.

The fix has two parts:

1. Use `.numpy()` to convert tensors to NumPy arrays (which matplotlib understands).
2. Wrap the computation in torch.no_grad() so that PyTorch doesn't try to build a graph for the plotting step.

In [ ]:
plt.plot(x.numpy(), y.numpy(), label="sin(x)")
with torch.no_grad():
    y_fit = a + b * x + c * x**2 + d * x**3
plt.plot(x.numpy(), y_fit.numpy(), label="Fitted polynomial")
plt.legend()
plt.title("Level 3: Autograd")
plt.show()


## Level 4: `nn` + `optim`

PyTorch provides two more abstractions that simplify the code further:

- **`torch.nn`**: defines model architectures (layers, activations). A `nn.Linear(in, out)` layer stores its own weights and computes $y = Wx + b$.
- **`torch.optim`**: implements optimization algorithms (SGD, Adam, etc.) so we don't manually update weights.

**Your task:** Replace the manual model and updates with `nn.Sequential` and `optim.SGD`.

*Hint:* The polynomial $a + bx + cx^2 + dx^3$ is a linear function of the features $(x, x^2, x^3)$. So we can use a single `nn.Linear(3, 1)` layer on the input tensor `[x, x², x³]`.

In [ ]:
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

# Prepare input tensor: columns are [x, x², x³]
p = torch.tensor([1, 2, 3])
# unsqueeze adds a dimension: (2000,) → (2000, 1), then pow broadcasts to (2000, 3)
xx = x.unsqueeze(-1).pow(p)

# TODO: define model using nn.Sequential
# model = torch.nn.Sequential(...)

# TODO: define loss function
# loss_fn = torch.nn.MSELoss(reduction='sum')

# TODO: define optimizer
# optimizer = torch.optim.SGD(model.parameters(), lr=...)

for t in range(2000):
    # TODO: forward pass, compute loss, backward, optimizer step
    pass

# Extract learned coefficients
linear_layer = model[0]
w = linear_layer.weight.detach().flatten()
bias = linear_layer.bias.item()

print(f"Result: y = {bias:.4f} + {w[0]:.4f}x + {w[1]:.4f}x² + {w[2]:.4f}x³")

with torch.no_grad():
    y_fit = model(xx)
plt.plot(x.numpy(), y.numpy(), label="sin(x)")
plt.plot(x.numpy(), y_fit.numpy(), label="Fitted polynomial")
plt.legend()
plt.title("Level 4: nn + optim")
plt.show()

## Summary

| Level | What you write | What PyTorch handles |
|---|---|---|
| 1. NumPy | Everything | Nothing |
| 2. Tensors | Everything (but with GPU-ready arrays) | Array operations |
| 3. Autograd | Forward pass + update rule | Gradient computation |
| 4. `nn` + `optim` | Model definition + training loop | Gradients, parameters, optimization |

Level 4 is the standard way to work in PyTorch. In the next exercise, we'll use `nn.Module` to define custom neural networks with hidden layers.